# Faruq-v3 ACMC -- static audit sebelum training

Mengaudit kandidat one-stage **Ambiguity-Conditioned Multilevel Classification Head** terhadap checkpoint D0 aktual. Notebook ini tidak membaca dataset, tidak membuka test, dan tidak melakukan training. Ia memastikan koreksi nol identik dengan D0, koreksi hanya mengubah skor klasifikasi, dan implementasi tidak memakai ROI/crop/top-K/decode box.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, shutil, subprocess, sys, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
if REPO.exists():
    shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
    if attempt == 3:
        raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
print('REPO:', REPO)

In [ ]:
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
))
D0_CHECKPOINT = require_project_artifact(
    PROJECT_ROOT, 'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt'
)
MODEL_YAML = REPO / 'configs/coffee_fg/models/yolo26n-p3.yaml'
OUTPUT = PROJECT_ROOT / 'experiments/faruq-v3-acmc-one-stage-v1/static_audit.json'
assert MODEL_YAML.is_file(), MODEL_YAML
print('PROJECT:', PROJECT_ROOT)
print('D0     :', D0_CHECKPOINT)
print('OUTPUT :', OUTPUT)

In [ ]:
from coffee_detector.ambiguity_multilevel.audit import static_ambiguity_multilevel_audit

result = static_ambiguity_multilevel_audit(
    MODEL_YAML, D0_CHECKPOINT, OUTPUT, nc=21, image_size=128,
    config={'hidden_dim': 64, 'context_kernel': 3, 'correction_scale': 1.0},
)
assert result['training_executed'] is False
assert result['dataset_accessed'] is False
assert result['test_images_accessed'] is False
print('PARAMETERS:', result['parameter_counts'])
print('ZERO DIFF :', result['zero_output_max_abs_diff'])
print('ACTIVE DIFF:', result['active_output_max_abs_diff'])
print('GATES     :', result['gates'])
print('DECISION  :', result['decision'])
print('SUMMARY   :', result['summary'])
assert result['decision'] == 'PASS', 'STOP: wiring ACMC tidak aman; jangan training.'
print('PASS: static gate terbuka. Kirim output ini sebelum screening seed 42.')